# Compute EmbeddingGemma 2 product embeddings

Replaces the fine-tuned (DoubleMLDeep) product embeddings with off-the-shelf embeddings from
[EmbeddingGemma 2](https://huggingface.co/google/embeddinggemma-2) (`google/embeddinggemma-2`, 740M parameters, 768-dimensional output with Matryoshka truncation to 512/256/128).

* One embedding per product (ASIN): the product text and image are time-invariant.
* `txtimg`: product text and product image interleaved into a single multimodal embedding (counterpart of the text+image model used in the main analysis).
* `txt`: product text only.
* Clustering task prefix `task: clustering | query: ` (symmetric task, see model card), mean pooling, L2-normalized, default vision budget of 280 soft tokens per image.
* Inference in `bfloat16` (recommended by the model card on hardware with native bf16 support; the CPU used here supports AMX-BF16).

The full 768-dimensional embeddings are stored in `../data/embeddings/`; Matryoshka truncation is applied when building the datasets (`01_1_create_dataset_txt_img.ipynb`, `01_2_create_dataset_txt.ipynb`).

Runtime: about 0.2 s per product for `txt` and 2.5 s per product for `txtimg` on a 4-core CPU (about 6 h in total). Intermediate results are cached in chunks in `../data/embeddings/_cache` (not part of the repository); the outputs below were produced from this cache.

In [1]:
import numpy as np
import pandas as pd

from utils.utils_gemma import (
    load_product_texts,
    load_product_image_bytes,
    compute_gemma_embeddings,
    embedding_metadata,
    save_embeddings,
)

dtype = "bfloat16"
num_threads = 4
data_dir = "../../main/data"
output_dir = "../data/embeddings"
cache_dir = "../data/embeddings/_cache"

dataset_paths = [
    f"{data_dir}/dataset_txt_only_False_embeddings_True_train.zip",
    f"{data_dir}/dataset_txt_only_False_embeddings_True_val.zip",
]
image_dir = f"{data_dir}/amzn_toys_monthly_avg_long_ffill_28_7_images"

## Products

In [2]:
df_products = load_product_texts(dataset_paths)
print(f"Number of products: {len(df_products)}")
print(df_products["text"].str.len().describe())
df_products.head()

Number of products: 7549
count    7549.000000
mean     1492.972314
std       775.271341
min       139.000000
25%       951.000000
50%      1412.000000
75%      1961.000000
max      5430.000000
Name: text, dtype: float64


,ASIN,text
0,1947335162,Catalyst Game Labs Battletech: Technical Reado...
1,1947494961,Gale Force Nine - Aliens: Another Glorious Day...
2,194749497X,"Gale Force Nine Table Top Game, Black, 8 x 26 ..."
3,9576243505,"Playmaker Toys 5"" Diecast Snow Plow Trucks (Bl..."
4,B00000IRXN,"Matchbox 5-Pack of 1:64 Scale Vehicles, 5 Toy ..."


In [3]:
image_bytes = load_product_image_bytes(image_dir, df_products["ASIN"])
print(f"Number of product images: {len(image_bytes)}")

Number of product images: 7549


## Text-only embeddings

In [4]:
df_emb_txt = compute_gemma_embeddings(
    df_products,
    modality="txt",
    dtype=dtype,
    batch_size=8,
    cache_dir=cache_dir,
    num_threads=num_threads,
)
save_embeddings(
    df_emb_txt,
    f"{output_dir}/embeddinggemma2_txt.parquet",
    metadata=embedding_metadata("txt", dtype, len(df_emb_txt)),
)
print(df_emb_txt.shape)

(7549, 768)


## Text + image embeddings

In [5]:
df_emb_txtimg = compute_gemma_embeddings(
    df_products,
    modality="txtimg",
    image_bytes=image_bytes,
    dtype=dtype,
    batch_size=8,
    cache_dir=cache_dir,
    num_threads=num_threads,
)
save_embeddings(
    df_emb_txtimg,
    f"{output_dir}/embeddinggemma2_txtimg.parquet",
    metadata=embedding_metadata("txtimg", dtype, len(df_emb_txtimg)),
)
print(df_emb_txtimg.shape)

(7549, 768)


## Sanity checks

In [6]:
for name, df_emb in [("txt", df_emb_txt), ("txtimg", df_emb_txtimg)]:
    norms = np.linalg.norm(df_emb.values, axis=1)
    print(
        f"{name}: shape={df_emb.shape}, finite={np.isfinite(df_emb.values).all()}, "
        f"norm min/max={norms.min():.6f}/{norms.max():.6f}"
    )

cos_txt_txtimg = (df_emb_txt.values * df_emb_txtimg.loc[df_emb_txt.index].values).sum(axis=1)
print("Cosine similarity text vs. text+image embedding of the same product:")
print(pd.Series(cos_txt_txtimg).describe())

txt: shape=(7549, 768), finite=True, norm min/max=1.000000/1.000000
txtimg: shape=(7549, 768), finite=True, norm min/max=1.000000/1.000000
Cosine similarity text vs. text+image embedding of the same product:
count    7549.000000
mean        0.950377
std         0.007620
min         0.916581
25%         0.946146
50%         0.951302
75%         0.955604
max         0.970661
dtype: float64


In [7]:
# Nearest neighbours (text+image embedding) of a few products
rng = np.random.default_rng(42)
texts = df_products.set_index("ASIN")["text"]
for asin in rng.choice(df_emb_txtimg.index, size=3, replace=False):
    sims = df_emb_txtimg.values @ df_emb_txtimg.loc[asin].values
    nn = df_emb_txtimg.index[np.argsort(-sims)[1:4]]
    print(f"{asin}: {texts[asin][:100]}")
    for n in nn:
        print(f"    -> {n}: {texts[n][:100]}")

B0BKL8HD3Y: WAKAKAC Compatible for 1/32 Wrangler Alloy Diecast Pull Back Model Car Collectible Gift with Light a
    -> B0BRKJY8VY: WAKAKAC Compatible for 1/32 Wrangler Alloy Diecast Pull Back Model Car Collectible Gift with Light a
    -> B0BKL6BP5L: WAKAKAC Compatible for 7 Door Wrangler Toy Car 1/32 Die-cast Pull Back Model Car with Sound and Ligh
    -> B09Y576JTK: BDTCTK Off-Road 7 Door for Wrangler Diecast Toy Car Model, 1/22 Scale Metal Alloy Vehicle for Kids B
B013RV1P48: KiNSFUN Die Cast Metal Food Truck 5 inch Pullback Action | Trucks | KiNSFUN Die Cast Metal Food Truc
    -> B003F62X24: KinsFun Pullback Action Ice Cream Vending Truck | Trucks | They'll hear you before they see you with
    -> B0B5G9YF53: KiNSMART United States Postal Mail Truck 1971 Jeep DJ-5B 5" Die Cast Model Toy Car 1:26 Scale | Truc
    -> B081S7X8XR: KiNSMART 1955 Chevy Stepside Pick-Up 2 Inch Pullback Action Trucks SetOf4 | Cars & Race Cars | KiNSM
B0BZM4RGMB: Hot Wheels Car Culture 2022 Exotic Envy Co